In [ ]:
from pathlib import Path
import json
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import xgboost as xgb

from scipy.stats import loguniform, randint, uniform

from sklearn.compose import (
    ColumnTransformer,
    make_column_selector
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)

print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgb.__version__)

In [ ]:
def find_project_root(start_path: Path) -> Path:
    """
    Find the project folder containing:
    data/processed/telco_churn_model.csv
    """
    start_path = start_path.resolve()

    for candidate in [start_path, *start_path.parents]:
        expected_file = (
            candidate
            / "data"
            / "processed"
            / "telco_churn_model.csv"
        )

        if expected_file.exists():
            return candidate

    raise FileNotFoundError(
        "Could not find "
        "data/processed/telco_churn_model.csv. "
        "Open the notebook from inside the "
        "customer-churn-project folder."
    )


project_root = find_project_root(Path.cwd())

model_data_path = (
    project_root
    / "data"
    / "processed"
    / "telco_churn_model.csv"
)

models_directory = project_root / "models"

model_output_directory = (
    project_root
    / "reports"
    / "model_results"
)

models_directory.mkdir(
    parents=True,
    exist_ok=True
)

model_output_directory.mkdir(
    parents=True,
    exist_ok=True
)

model_df = pd.read_csv(model_data_path)

print("Project root:", project_root)
print("Dataset path:", model_data_path)
print("Dataset shape:", model_df.shape)

display(model_df.head())

In [ ]:
target_column = "churn_value"

forbidden_columns = {
    "customerid",
    "churn_label",
    "churn_score",
    "churn_reason"
}

assert target_column in model_df.columns

assert model_df[target_column].isin(
    [0, 1]
).all()

assert (
    model_df[target_column]
    .isna()
    .sum()
    == 0
)

assert not forbidden_columns.intersection(
    model_df.columns
), (
    "Leakage or identifier columns found: "
    f"{forbidden_columns.intersection(model_df.columns)}"
)

X = model_df.drop(
    columns=[target_column]
).copy()

y = model_df[
    target_column
].astype(int).copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Churn rate:", round(y.mean(), 4))

In [ ]:
X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print(
    "Training churn rate:",
    round(y_train.mean(), 4)
)

print(
    "Test churn rate:",
    round(y_test.mean(), 4)
)

In [ ]:
numeric_selector = make_column_selector(
    dtype_include=np.number
)

categorical_selector = make_column_selector(
    dtype_exclude=np.number
)


def build_linear_preprocessor():
    """
    Preprocessing for Logistic Regression.
    """

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "scaler",
                StandardScaler()
            )
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    drop="if_binary"
                )
            )
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_selector
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_selector
            )
        ],
        remainder="drop"
    )


def build_tree_preprocessor():
    """
    Preprocessing for Random Forest and XGBoost.
    Scaling is unnecessary for tree models.
    """

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median")
            )
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    drop="if_binary"
                )
            )
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_selector
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_selector
            )
        ],
        remainder="drop"
    )

In [ ]:
detected_numeric = numeric_selector(X_train)

detected_categorical = categorical_selector(
    X_train
)

detected_columns = set(
    detected_numeric
    + detected_categorical
)

missing_columns = (
    set(X_train.columns)
    - detected_columns
)

print("Numeric columns:")
print(detected_numeric)

print("\nCategorical columns:")
print(detected_categorical)

print("\nDetected:", len(detected_columns))
print("Expected:", X_train.shape[1])
print("Missed columns:", missing_columns)

assert len(detected_columns) == X_train.shape[1]
assert not missing_columns

probe_preprocessor = (
    build_linear_preprocessor()
)

probe_matrix = (
    probe_preprocessor
    .fit_transform(X_train.iloc[:100])
)

print("\nPreprocessor test passed.")
print(
    "Input shape:",
    X_train.iloc[:100].shape
)
print(
    "Transformed shape:",
    probe_matrix.shape
)

In [ ]:
cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "recall": "recall",
    "precision": "precision",
    "f1": "f1"
}

SEARCH_RANDOM_STATE = 42

N_ITER_LOGISTIC = 20
N_ITER_RANDOM_FOREST = 25
N_ITER_XGBOOST = 30

In [ ]:
def detect_xgboost_device():
    """
    Test whether XGBoost can train using CUDA.
    """

    X_probe = np.array(
        [
            [0.0, 0.0],
            [0.0, 1.0],
            [1.0, 0.0],
            [1.0, 1.0],
            [2.0, 0.0],
            [2.0, 1.0]
        ],
        dtype=np.float32
    )

    y_probe = np.array(
        [0, 0, 0, 1, 1, 1],
        dtype=np.int32
    )

    probe = XGBClassifier(
        n_estimators=5,
        max_depth=2,
        tree_method="hist",
        device="cuda",
        eval_metric="logloss",
        random_state=42,
        n_jobs=1,
        verbosity=0
    )

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            probe.fit(X_probe, y_probe)

        config = json.loads(
            probe.get_booster().save_config()
        )

        actual_device = (
            config["learner"]
            ["generic_param"]
            ["device"]
        )

        if str(actual_device).startswith("cuda"):
            print(
                "XGBoost GPU test passed:",
                actual_device
            )
            return "cuda"

        print(
            "CUDA was not recorded. "
            "Using CPU."
        )

        return "cpu"

    except Exception as error:
        print(
            "XGBoost GPU test failed. "
            "Using CPU."
        )
        print("Reason:", error)

        return "cpu"


XGBOOST_DEVICE = detect_xgboost_device()

print(
    "Selected XGBoost device:",
    XGBOOST_DEVICE
)

In [ ]:
tuning_logistic_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            build_linear_preprocessor()
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=5000,
                random_state=42
            )
        )
    ]
)


logistic_parameter_distributions = {
    "classifier__solver": [
        "lbfgs",
        "newton-cholesky"
    ],

    # Regularization strength
    "classifier__C": loguniform(
        1e-3,
        100
    ),

    # l1_ratio = 0 means L2 regularization
    "classifier__l1_ratio": [0.0],

    "classifier__class_weight": [
        None,
        "balanced"
    ]
}


logistic_search = RandomizedSearchCV(
    estimator=tuning_logistic_pipeline,

    param_distributions=(
        logistic_parameter_distributions
    ),

    n_iter=N_ITER_LOGISTIC,

    scoring=scoring,

    # Choose the model having the best PR-AUC
    refit="pr_auc",

    cv=cross_validation,

    random_state=SEARCH_RANDOM_STATE,

    # Logistic Regression can parallelize
    # different search fits across CPU cores
    n_jobs=-1,

    verbose=1,

    return_train_score=True,

    # Stop immediately if a parameter combination fails
    error_score="raise"
)

print("Logistic Regression search created.")

In [ ]:
start_time = time.perf_counter()

logistic_search.fit(
    X_train,
    y_train
)

logistic_tuning_time = (
    time.perf_counter() - start_time
)

print(
    f"Logistic tuning time: "
    f"{logistic_tuning_time:.2f} seconds"
)

print(
    "\nBest Logistic Regression CV PR-AUC:"
)

print(
    round(logistic_search.best_score_, 4)
)

print("\nBest parameters:")

for parameter, value in (
    logistic_search.best_params_.items()
):
    print(f"{parameter}: {value}")

In [ ]:
tuning_random_forest_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            build_tree_preprocessor()
        ),
        (
            "classifier",
            RandomForestClassifier(
                random_state=42,

                # Each Random Forest fit uses all CPU cores
                n_jobs=-1
            )
        )
    ]
)


random_forest_parameter_distributions = {
    "classifier__n_estimators": randint(
        200,
        901
    ),

    "classifier__max_depth": [
        None,
        4,
        6,
        8,
        10,
        12,
        16,
        20
    ],

    "classifier__min_samples_split": randint(
        2,
        21
    ),

    "classifier__min_samples_leaf": randint(
        1,
        11
    ),

    "classifier__max_features": [
        "sqrt",
        "log2",
        0.5,
        0.75,
        None
    ],

    "classifier__class_weight": [
        None,
        "balanced",
        "balanced_subsample"
    ]
}


random_forest_search = RandomizedSearchCV(
    estimator=tuning_random_forest_pipeline,

    param_distributions=(
        random_forest_parameter_distributions
    ),

    n_iter=N_ITER_RANDOM_FOREST,

    scoring=scoring,

    refit="pr_auc",

    cv=cross_validation,

    random_state=SEARCH_RANDOM_STATE,

    # Keep outer search sequential because
    # Random Forest already uses all CPU cores
    n_jobs=1,

    verbose=1,

    return_train_score=True,

    error_score="raise"
)

print("Random Forest search created.")

In [ ]:
start_time = time.perf_counter()

random_forest_search.fit(
    X_train,
    y_train
)

random_forest_tuning_time = (
    time.perf_counter() - start_time
)

print(
    f"Random Forest tuning time: "
    f"{random_forest_tuning_time:.2f} seconds"
)

print(
    "\nBest Random Forest CV PR-AUC:"
)

print(
    round(
        random_forest_search.best_score_,
        4
    )
)

print("\nBest parameters:")

for parameter, value in (
    random_forest_search.best_params_.items()
):
    print(f"{parameter}: {value}")

In [ ]:
negative_customers = int(
    (y_train == 0).sum()
)

positive_customers = int(
    (y_train == 1).sum()
)

class_ratio = (
    negative_customers
    / positive_customers
)

print(
    "Non-churned training customers:",
    negative_customers
)

print(
    "Churned training customers:",
    positive_customers
)

print(
    "Negative-to-positive ratio:",
    round(class_ratio, 4)
)

In [ ]:
tuning_xgboost_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            build_tree_preprocessor()
        ),
        (
            "classifier",
            XGBClassifier(
                objective="binary:logistic",

                tree_method="hist",

                device=XGBOOST_DEVICE,

                eval_metric="logloss",

                random_state=42,

                # Avoid CPU oversubscription
                n_jobs=1,

                verbosity=0
            )
        )
    ]
)


xgboost_parameter_distributions = {
    "classifier__n_estimators": randint(
        200,
        901
    ),

    "classifier__learning_rate": loguniform(
        0.01,
        0.20
    ),

    "classifier__max_depth": randint(
        2,
        9
    ),

    "classifier__min_child_weight": randint(
        1,
        11
    ),

    # Samples values from 0.65 to 1.00
    "classifier__subsample": uniform(
        0.65,
        0.35
    ),

    # Samples values from 0.65 to 1.00
    "classifier__colsample_bytree": uniform(
        0.65,
        0.35
    ),

    "classifier__gamma": uniform(
        0.0,
        5.0
    ),

    "classifier__reg_alpha": loguniform(
        1e-4,
        10
    ),

    "classifier__reg_lambda": loguniform(
        1e-3,
        20
    ),

    "classifier__scale_pos_weight": [
        1.0,
        1.25,
        1.5,
        2.0,
        class_ratio
    ]
}


xgboost_search = RandomizedSearchCV(
    estimator=tuning_xgboost_pipeline,

    param_distributions=(
        xgboost_parameter_distributions
    ),

    n_iter=N_ITER_XGBOOST,

    scoring=scoring,

    refit="pr_auc",

    cv=cross_validation,

    random_state=SEARCH_RANDOM_STATE,

    # Run one fit at a time so multiple
    # fits do not compete for the same GPU
    n_jobs=1,

    verbose=1,

    return_train_score=True,

    error_score="raise"
)

print(
    "XGBoost search created using device:",
    XGBOOST_DEVICE
)

In [ ]:
start_time = time.perf_counter()

xgboost_search.fit(
    X_train,
    y_train
)

xgboost_tuning_time = (
    time.perf_counter() - start_time
)

print(
    f"XGBoost tuning time: "
    f"{xgboost_tuning_time:.2f} seconds"
)

print(
    "Selected XGBoost device:",
    XGBOOST_DEVICE
)

print(
    "\nBest XGBoost CV PR-AUC:"
)

print(
    round(
        xgboost_search.best_score_,
        4
    )
)

print("\nBest parameters:")

for parameter, value in (
    xgboost_search.best_params_.items()
):
    print(f"{parameter}: {value}")

In [ ]:
best_xgboost_classifier = (
    xgboost_search
    .best_estimator_
    .named_steps["classifier"]
)

xgboost_configuration = json.loads(
    best_xgboost_classifier
    .get_booster()
    .save_config()
)

trained_xgboost_device = (
    xgboost_configuration["learner"]
    ["generic_param"]
    ["device"]
)

print(
    "Device recorded by trained XGBoost model:",
    trained_xgboost_device
)

In [ ]:
def extract_best_search_result(
    model_name,
    search_object,
    tuning_time
):
    """
    Extract the cross-validation results belonging
    to the best parameter combination.
    """

    best_index = search_object.best_index_

    results = search_object.cv_results_

    return {
        "model": model_name,

        "pr_auc_mean": results[
            "mean_test_pr_auc"
        ][best_index],

        "pr_auc_std": results[
            "std_test_pr_auc"
        ][best_index],

        "roc_auc_mean": results[
            "mean_test_roc_auc"
        ][best_index],

        "roc_auc_std": results[
            "std_test_roc_auc"
        ][best_index],

        "recall_mean": results[
            "mean_test_recall"
        ][best_index],

        "precision_mean": results[
            "mean_test_precision"
        ][best_index],

        "f1_mean": results[
            "mean_test_f1"
        ][best_index],

        "training_pr_auc": results[
            "mean_train_pr_auc"
        ][best_index],

        "tuning_time_seconds": tuning_time
    }

In [ ]:
tuned_model_comparison = pd.DataFrame(
    [
        extract_best_search_result(
            model_name="Logistic Regression",
            search_object=logistic_search,
            tuning_time=logistic_tuning_time
        ),

        extract_best_search_result(
            model_name="Random Forest",
            search_object=random_forest_search,
            tuning_time=random_forest_tuning_time
        ),

        extract_best_search_result(
            model_name="XGBoost",
            search_object=xgboost_search,
            tuning_time=xgboost_tuning_time
        )
    ]
)


tuned_model_comparison = (
    tuned_model_comparison
    .sort_values(
        by=[
            "pr_auc_mean",
            "roc_auc_mean"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    tuned_model_comparison.round(4)
)

In [ ]:
baseline_path = (
    model_output_directory
    / "all_models_cv_comparison.csv"
)


if baseline_path.exists():

    baseline_results = pd.read_csv(
        baseline_path
    )

    baseline_results["model"] = (
        baseline_results["model"]
        .replace(
            {
                "XGBoost GPU": "XGBoost"
            }
        )
    )

    baseline_results = baseline_results[
        baseline_results["model"].isin(
            [
                "Logistic Regression",
                "Random Forest",
                "XGBoost"
            ]
        )
    ][
        [
            "model",
            "pr_auc_mean",
            "roc_auc_mean",
            "recall_mean",
            "precision_mean",
            "f1_mean"
        ]
    ].copy()

    baseline_results["stage"] = "Untuned"


    tuned_results_for_comparison = (
        tuned_model_comparison[
            [
                "model",
                "pr_auc_mean",
                "roc_auc_mean",
                "recall_mean",
                "precision_mean",
                "f1_mean"
            ]
        ]
        .copy()
    )

    tuned_results_for_comparison[
        "stage"
    ] = "Tuned"


    untuned_vs_tuned = pd.concat(
        [
            baseline_results,
            tuned_results_for_comparison
        ],
        ignore_index=True
    )


    display(
        untuned_vs_tuned
        .sort_values(
            by=["model", "stage"]
        )
        .round(4)
    )

else:

    untuned_vs_tuned = pd.DataFrame()

    print(
        "all_models_cv_comparison.csv "
        "was not found."
    )

    print(
        "The tuned-model comparison is still valid."
    )

In [ ]:
metrics_to_plot = [
    "precision_mean",
    "recall_mean",
    "f1_mean",
    "roc_auc_mean",
    "pr_auc_mean"
]


plot_data = (
    tuned_model_comparison
    .set_index("model")[
        metrics_to_plot
    ]
    .T
)


plot_data.index = [
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC",
    "PR-AUC"
]


plot_data.plot(
    kind="bar",
    figsize=(11, 6)
)

plt.title(
    "Tuned Model Cross-Validation Comparison"
)

plt.xlabel("Evaluation Metric")
plt.ylabel("Mean Cross-Validation Score")

plt.ylim(0, 1)

plt.xticks(rotation=0)

plt.legend(
    title="Model",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

In [ ]:
best_estimators = {
    "Logistic Regression": (
        logistic_search.best_estimator_
    ),

    "Random Forest": (
        random_forest_search.best_estimator_
    ),

    "XGBoost": (
        xgboost_search.best_estimator_
    )
}


selected_tuned_model_name = (
    tuned_model_comparison
    .loc[0, "model"]
)


selected_tuned_model = (
    best_estimators[
        selected_tuned_model_name
    ]
)


print(
    "Selected tuned model by "
    "cross-validation PR-AUC:"
)

print(selected_tuned_model_name)

In [ ]:
joblib.dump(
    logistic_search.best_estimator_,

    models_directory
    / "tuned_logistic_regression.joblib"
)


joblib.dump(
    random_forest_search.best_estimator_,

    models_directory
    / "tuned_random_forest.joblib"
)


joblib.dump(
    xgboost_search.best_estimator_,

    models_directory
    / "tuned_xgboost_gpu.joblib"
)


joblib.dump(
    selected_tuned_model,

    models_directory
    / "best_tuned_cv_model.joblib"
)


print("Tuned models saved successfully.")

In [ ]:
best_parameters = {
    "logistic_regression": (
        logistic_search.best_params_
    ),

    "random_forest": (
        random_forest_search.best_params_
    ),

    "xgboost": (
        xgboost_search.best_params_
    ),

    "selected_model": (
        selected_tuned_model_name
    ),

    "selection_metric": (
        "mean cross-validation PR-AUC"
    ),

    "xgboost_device": (
        XGBOOST_DEVICE
    ),

    "random_state": (
        SEARCH_RANDOM_STATE
    )
}


parameters_path = (
    models_directory
    / "tuned_model_parameters.json"
)


with open(
    parameters_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        best_parameters,
        file,
        indent=4,
        default=str
    )

In [ ]:
tuned_model_comparison.to_csv(
    model_output_directory
    / "tuned_models_cv_comparison.csv",

    index=False
)


if not untuned_vs_tuned.empty:

    untuned_vs_tuned.to_csv(
        model_output_directory
        / "untuned_vs_tuned_comparison.csv",

        index=False
    )

In [ ]:
pd.DataFrame(
    logistic_search.cv_results_
).to_csv(
    model_output_directory
    / "logistic_tuning_all_trials.csv",

    index=False
)


pd.DataFrame(
    random_forest_search.cv_results_
).to_csv(
    model_output_directory
    / "random_forest_tuning_all_trials.csv",

    index=False
)


pd.DataFrame(
    xgboost_search.cv_results_
).to_csv(
    model_output_directory
    / "xgboost_tuning_all_trials.csv",

    index=False
)


print("Models saved to:")
print(models_directory)

print("\nTuning results saved to:")
print(model_output_directory)

In [ ]:
final_tuning_summary = (
    tuned_model_comparison[
        [
            "model",
            "pr_auc_mean",
            "pr_auc_std",
            "roc_auc_mean",
            "recall_mean",
            "precision_mean",
            "f1_mean",
            "training_pr_auc",
            "tuning_time_seconds"
        ]
    ]
    .copy()
)


display(
    final_tuning_summary.round(4)
)


print(
    "\nBest tuned model:",
    selected_tuned_model_name
)

print(
    "Best CV PR-AUC:",
    round(
        tuned_model_comparison
        .loc[0, "pr_auc_mean"],
        4
    )
)

In [ ]:
model_generalization_summary = (
    tuned_model_comparison[
        [
            "model",
            "training_pr_auc",
            "pr_auc_mean",
            "pr_auc_std",
            "roc_auc_mean",
            "recall_mean",
            "precision_mean",
            "f1_mean"
        ]
    ]
    .copy()
)


model_generalization_summary[
    "train_validation_pr_auc_gap"
] = (
    model_generalization_summary[
        "training_pr_auc"
    ]
    -
    model_generalization_summary[
        "pr_auc_mean"
    ]
)


model_generalization_summary[
    "possible_overfitting"
] = np.where(
    model_generalization_summary[
        "train_validation_pr_auc_gap"
    ] > 0.10,
    "Review",
    "Acceptable"
)


display(
    model_generalization_summary.round(4)
)

In [ ]:
if not untuned_vs_tuned.empty:

    comparison_pivot = (
        untuned_vs_tuned
        .pivot(
            index="model",
            columns="stage",
            values=[
                "pr_auc_mean",
                "roc_auc_mean",
                "recall_mean",
                "precision_mean",
                "f1_mean"
            ]
        )
    )

    display(
        comparison_pivot.round(4)
    )

else:

    print(
        "Untuned model results were not found."
    )

    print(
        "The tuned-model comparison is still available."
    )

In [ ]:
if not untuned_vs_tuned.empty:

    untuned_metrics = (
        untuned_vs_tuned[
            untuned_vs_tuned["stage"] == "Untuned"
        ]
        .set_index("model")
    )


    tuned_metrics = (
        untuned_vs_tuned[
            untuned_vs_tuned["stage"] == "Tuned"
        ]
        .set_index("model")
    )


    common_models = (
        untuned_metrics.index
        .intersection(tuned_metrics.index)
    )


    tuning_improvement_summary = pd.DataFrame(
        {
            "model": common_models,

            "untuned_pr_auc": (
                untuned_metrics.loc[
                    common_models,
                    "pr_auc_mean"
                ].values
            ),

            "tuned_pr_auc": (
                tuned_metrics.loc[
                    common_models,
                    "pr_auc_mean"
                ].values
            ),

            "untuned_roc_auc": (
                untuned_metrics.loc[
                    common_models,
                    "roc_auc_mean"
                ].values
            ),

            "tuned_roc_auc": (
                tuned_metrics.loc[
                    common_models,
                    "roc_auc_mean"
                ].values
            )
        }
    )


    tuning_improvement_summary[
        "pr_auc_improvement"
    ] = (
        tuning_improvement_summary[
            "tuned_pr_auc"
        ]
        -
        tuning_improvement_summary[
            "untuned_pr_auc"
        ]
    )


    tuning_improvement_summary[
        "roc_auc_improvement"
    ] = (
        tuning_improvement_summary[
            "tuned_roc_auc"
        ]
        -
        tuning_improvement_summary[
            "untuned_roc_auc"
        ]
    )


    tuning_improvement_summary = (
        tuning_improvement_summary
        .sort_values(
            "tuned_pr_auc",
            ascending=False
        )
        .reset_index(drop=True)
    )


    display(
        tuning_improvement_summary.round(4)
    )

else:

    tuning_improvement_summary = pd.DataFrame()

    print(
        "Improvement table could not be created "
        "because untuned results are unavailable."
    )

In [ ]:
if not untuned_vs_tuned.empty:

    pr_auc_plot_data = (
        untuned_vs_tuned
        .pivot(
            index="model",
            columns="stage",
            values="pr_auc_mean"
        )
    )


    pr_auc_plot_data.plot(
        kind="bar",
        figsize=(9, 5)
    )


    plt.title(
        "Untuned vs Tuned Cross-Validation PR-AUC"
    )

    plt.xlabel("Model")
    plt.ylabel("Mean Cross-Validation PR-AUC")

    plt.ylim(0, 1)

    plt.xticks(rotation=0)

    plt.legend(
        title="Training Stage"
    )

    plt.tight_layout()
    plt.show()

else:

    print(
        "Untuned-versus-tuned plot skipped."
    )

In [ ]:
selected_model_parameter_mapping = {
    "Logistic Regression": (
        logistic_search.best_params_
    ),

    "Random Forest": (
        random_forest_search.best_params_
    ),

    "XGBoost": (
        xgboost_search.best_params_
    )
}


selected_tuned_parameters = (
    selected_model_parameter_mapping[
        selected_tuned_model_name
    ]
)


print(
    "Selected tuned model:",
    selected_tuned_model_name
)

print(
    "\nSelected model parameters:"
)


for parameter, value in (
    selected_tuned_parameters.items()
):

    print(
        f"{parameter}: {value}"
    )

In [ ]:
selected_model_row = (
    tuned_model_comparison
    .iloc[0]
)


model_selection_summary = pd.DataFrame(
    [
        {
            "selected_model": (
                selected_tuned_model_name
            ),

            "selection_metric": (
                "Mean cross-validation PR-AUC"
            ),

            "cv_pr_auc_mean": (
                selected_model_row[
                    "pr_auc_mean"
                ]
            ),

            "cv_pr_auc_std": (
                selected_model_row[
                    "pr_auc_std"
                ]
            ),

            "cv_roc_auc_mean": (
                selected_model_row[
                    "roc_auc_mean"
                ]
            ),

            "cv_recall_mean": (
                selected_model_row[
                    "recall_mean"
                ]
            ),

            "cv_precision_mean": (
                selected_model_row[
                    "precision_mean"
                ]
            ),

            "cv_f1_mean": (
                selected_model_row[
                    "f1_mean"
                ]
            ),

            "training_pr_auc": (
                selected_model_row[
                    "training_pr_auc"
                ]
            ),

            "train_validation_gap": (
                selected_model_row[
                    "training_pr_auc"
                ]
                -
                selected_model_row[
                    "pr_auc_mean"
                ]
            ),

            "test_set_used_for_selection": False
        }
    ]
)


display(
    model_selection_summary.round(4)
)

In [ ]:
final_tuning_summary.to_csv(
    model_output_directory
    / "final_tuning_summary.csv",

    index=False
)


model_generalization_summary.to_csv(
    model_output_directory
    / "model_generalization_summary.csv",

    index=False
)


model_selection_summary.to_csv(
    model_output_directory
    / "tuned_model_selection_summary.csv",

    index=False
)


if not tuning_improvement_summary.empty:

    tuning_improvement_summary.to_csv(
        model_output_directory
        / "tuning_improvement_summary.csv",

        index=False
    )


print(
    "Notebook 4 summary tables saved successfully."
)

In [ ]:
def convert_to_python_type(value):
    """
    Convert NumPy values into standard Python values
    so they can be saved as JSON.
    """

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    if isinstance(value, np.bool_):
        return bool(value)

    if isinstance(value, np.ndarray):
        return value.tolist()

    return value

notebook_4_metadata = {
    "notebook": "04_hyperparameter_tuning",

    "target_column": target_column,

    "training_rows": int(
        X_train.shape[0]
    ),

    "test_rows_reserved": int(
        X_test.shape[0]
    ),

    "number_of_input_features": int(
        X_train.shape[1]
    ),

    "cross_validation_folds": int(
        cross_validation.n_splits
    ),

    "primary_selection_metric": (
        "average_precision / PR-AUC"
    ),

    "selected_model": (
        selected_tuned_model_name
    ),

    "selected_model_parameters": {
        key: convert_to_python_type(value)

        for key, value in (
            selected_tuned_parameters.items()
        )
    },

    "selected_model_cv_metrics": {
        "pr_auc_mean": float(
            selected_model_row[
                "pr_auc_mean"
            ]
        ),

        "pr_auc_std": float(
            selected_model_row[
                "pr_auc_std"
            ]
        ),

        "roc_auc_mean": float(
            selected_model_row[
                "roc_auc_mean"
            ]
        ),

        "recall_mean": float(
            selected_model_row[
                "recall_mean"
            ]
        ),

        "precision_mean": float(
            selected_model_row[
                "precision_mean"
            ]
        ),

        "f1_mean": float(
            selected_model_row[
                "f1_mean"
            ]
        ),

        "training_pr_auc": float(
            selected_model_row[
                "training_pr_auc"
            ]
        )
    },

    "xgboost_device": (
        XGBOOST_DEVICE
    ),

    "test_set_evaluated": False,

    "next_notebook": (
        "05_final_tuned_model.ipynb"
    )
}

In [ ]:
notebook_4_metadata_path = (
    models_directory
    / "hyperparameter_tuning_metadata.json"
)


with open(
    notebook_4_metadata_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        notebook_4_metadata,
        file,
        indent=4
    )


print(
    "Metadata saved to:"
)

print(
    notebook_4_metadata_path
)

In [ ]:
required_model_artifacts = [
    models_directory
    / "tuned_logistic_regression.joblib",

    models_directory
    / "tuned_random_forest.joblib",

    models_directory
    / "tuned_xgboost_gpu.joblib",

    models_directory
    / "best_tuned_cv_model.joblib",

    models_directory
    / "tuned_model_parameters.json",

    models_directory
    / "hyperparameter_tuning_metadata.json"
]


required_result_artifacts = [
    model_output_directory
    / "tuned_models_cv_comparison.csv",

    model_output_directory
    / "final_tuning_summary.csv",

    model_output_directory
    / "model_generalization_summary.csv",

    model_output_directory
    / "tuned_model_selection_summary.csv"
]


artifact_check = pd.DataFrame(
    [
        {
            "artifact": path.name,
            "location": str(path.parent),
            "exists": path.exists()
        }

        for path in (
            required_model_artifacts
            +
            required_result_artifacts
        )
    ]
)


display(
    artifact_check
)


missing_artifacts = artifact_check[
    artifact_check["exists"] == False
]


if missing_artifacts.empty:

    print(
        "\nAll required Notebook 4 "
        "artifacts were created successfully."
    )

else:

    print(
        "\nSome artifacts are missing:"
    )

    display(
        missing_artifacts
    )

In [ ]:
saved_best_model_path = (
    models_directory
    / "best_tuned_cv_model.joblib"
)


reloaded_best_model = joblib.load(
    saved_best_model_path
)


print(
    "Reloaded model type:"
)

print(
    type(reloaded_best_model)
)


print(
    "\nPipeline steps:"
)

print(
    reloaded_best_model.named_steps.keys()
)

print("=" * 60)

print(
    "NOTEBOOK 4 — HYPERPARAMETER TUNING COMPLETE"
)

print("=" * 60)

print(
    f"Selected model: "
    f"{selected_tuned_model_name}"
)

print(
    f"Mean CV PR-AUC: "
    f"{selected_model_row['pr_auc_mean']:.4f}"
)

print(
    f"CV PR-AUC standard deviation: "
    f"{selected_model_row['pr_auc_std']:.4f}"
)

print(
    f"Mean CV ROC-AUC: "
    f"{selected_model_row['roc_auc_mean']:.4f}"
)

print(
    "\nThe test set remains reserved."
)

print(
    "Threshold tuning and final evaluation "
    "will be performed in Notebook 5."
)

print(
    "\nNext notebook:"
)

print(
    "05_final_tuned_model.ipynb"
)